# TUM-VIE Dataset Explorer (Tonic)

This notebook explores the TUM Stereo Visual-Inertial Event Dataset as exposed by Tonic via `tonic.datasets.TUMVIE`.

Goal:
- Inspect recording availability and schema
- Visualize event-frame density for left/right cameras
- Summarize IMU and mocap target statistics

Note: `tonic.datasets.tum_vie` is a module namespace; `TUMVIE` is the dataset class used below.

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from tonic.datasets import TUMVIE
from tonic import transforms

DATA_ROOT = Path('../data').resolve()
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT)
print('recordings =', len(TUMVIE.recordings))
print(TUMVIE.recordings)

In [ ]:
recording = 'mocap-6dof'
n_time_bins = 32

transform = transforms.Compose([
    transforms.ToFrame(sensor_size=TUMVIE.sensor_size, n_time_bins=n_time_bins),
])

dataset = TUMVIE(save_to=str(DATA_ROOT), recording=recording, transform=transform)
print('dataset length =', len(dataset))

In [ ]:
sample_idx = 0
data_dict, target_dict = dataset[sample_idx]

print('data keys   :', sorted(data_dict.keys()))
print('target keys :', sorted(target_dict.keys()))
print('events_left shape  :', data_dict['events_left'].shape)
print('events_right shape :', data_dict['events_right'].shape)
print('imu shape          :', np.asarray(data_dict['imu']).shape)
print('mocap shape        :', np.asarray(target_dict['mocap']).shape)
print('mocap first row    :', np.asarray(target_dict['mocap'])[0])

In [ ]:
left = np.asarray(data_dict['events_left'])  # [T, C, H, W]
right = np.asarray(data_dict['events_right'])

left_density = left.sum(axis=(1, 2, 3))
right_density = right.sum(axis=(1, 2, 3))

plt.figure(figsize=(10, 3))
plt.plot(left_density, label='left frame density')
plt.plot(right_density, label='right frame density')
plt.title(f'Event density over time bins ({recording})')
plt.xlabel('time bin')
plt.ylabel('sum(events)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.ravel()):
    t = int(i * max(1, left.shape[0] // 8))
    t = min(t, left.shape[0] - 1)
    frame = left[t].sum(axis=0)
    ax.imshow(frame, cmap='magma')
    ax.set_title(f'left t={t}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
def collect_quick_stats(ds, n=16):
    imu_lengths = []
    mocap_means = []
    frame_totals = []
    n = min(n, len(ds))
    for i in range(n):
        x, y = ds[i]
        imu_lengths.append(np.asarray(x['imu']).shape[0])
        mocap = np.asarray(y['mocap'])
        mocap_means.append(mocap.mean(axis=0))
        frame_totals.append(np.asarray(x['events_left']).sum())
    return {
        'samples': n,
        'imu_len_mean': float(np.mean(imu_lengths)),
        'imu_len_std': float(np.std(imu_lengths)),
        'frame_total_mean': float(np.mean(frame_totals)),
        'mocap_mean': np.mean(np.stack(mocap_means, axis=0), axis=0),
    }

stats = collect_quick_stats(dataset, n=10)
stats

## Next Steps

- Switch to notebooks in `research/paper` for model comparison experiments.
- Try alternative recordings: `running-easy`, `bike-easy`, `loop-floor0`.
- Reuse this preprocessing path when benchmarking `IMUConditionedVisualSNN` and Kalman-style fusion models.